# 09 · End-to-end query cost of source information

Compare **always exact**, **A18 + exact fallback**, and **B24 + exact fallback** using the completed 08d run.
No training, new model selection, recalibration, merge-policy change, or test-tuned routing is performed.
The experiment measures the current Java runtime. It does not run Spark SQL scans or claim cold remote-storage latency.

## Before running

1. Use a **classic dedicated ML cluster with a JDK (`java` and `javac`)**. Run the dependency installation cell below before importing Python modules. It installs LightGBM 4.6.0, matching the saved 08d run.
2. Build the current team JAR from the repository root: `cd SynopsisLake && mvn package`. Upload the resulting
   `target/SynopsisLake-0.0.1-SNAPSHOT.jar` to a location readable by this notebook and set `jar_path`.
   Building is local; no deployment workflow or token is required. Supply `math_jar` if commons-math3 is not discoverable.
3. Run the installation cell, then the imports/configuration cell to create widgets. **Set `run_dir` to the completed 08d directory**, not the exported compact ZIP.
   It must contain `run.json`, `splits.csv`, `gate_details.csv`, and both variants' complete `models/` folders
   (weights, schemas, cohorts and saved test predictions). No personal workspace path is committed here.
4. Point `raw_dir` and `raw_files` at the **same original raw data** that produced 08d's answers.
   Inputs default to `run_dir/inputs`; use `data_dir`/`histogram_dir` only for relocated matching inputs.
   Input SHA256 identities are checked. Missing or incompatible artifacts fail explicitly.

**Smoke defaults:** seed `0`, nearest saved merge rate to `0.5`, at most `20` test queries, `3` repeats,
policy `empirical`, histogram mode `resident`. The nearest rate is recorded; no rate is selected using test quality.
These defaults verify correctness and produce a small diagnostic, not a stable performance conclusion.

**Larger exploratory run:** `seeds=0,1,2`, `merge_rates=all`, `max_queries=100` (or `0` for every saved test query),
`repeats=5`, `cache_modes=resident,reload`. Run `policies=grid_empirical` separately, or list policies to report them separately.
The full-data/all-rate run can be expensive: every exact request and fallback scans every raw point.

In [ ]:
%pip install lightgbm==4.6.0

In [ ]:
import csv
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import lightgbm as lgb
import numpy as np
import pandas as pd

VARIANTS = ('no_sa_contrast', 'no_sa_all_source')


def default_repo_root():
    try:
        name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        root = name.split('/ml/notebooks/')[0]
        return root if root.startswith('/Workspace/') else '/Workspace' + root
    except (NameError, AttributeError):
        return str(Path.cwd())


def default_output_root():
    try:
        user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
        return '/Workspace/Users/' + user + '/synopsislake-results'
    except (NameError, AttributeError):
        return str(Path(tempfile.gettempdir()) / 'synopsislake-results')


DEFAULTS = dict(
    run_dir='', repo_root=default_repo_root(), output_root=default_output_root(),
    data_dir='', histogram_dir='',
    raw_dir='/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/range_query',
    raw_files='0.csv,1.csv,2.csv,3.csv',
    jar_path='/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/jars/synopsislake.jar',
    math_jar='', java='java', javac='javac', seeds='0', merge_rates='0.5',
    policies='empirical', max_queries='20', repeats='3', warmup_queries='3',
    tree_budget='', cache_modes='resident', num_threads='2', random_seed='20261112')
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_setting = dbutils.widgets.get
except NameError:
    def get_setting(key):
        return os.environ.get(key.upper(), DEFAULTS[key])

## What is included in each request?

- Full outer request time, histogram/source loading when requested, histogram estimation, feature extraction,
  model inference, acceptance decision and actual raw-point scan on fallback.
- **Raw points and model weights are resident in every mode.** Their loading, validation and preprocessing are
  reported separately in `setup.csv` and `run.json`, along with current training/calibration time (both zero)
  and original 08d fitting/evaluation metadata. Process wall time is not request latency.
- `resident`: histogram contexts are preloaded. `reload`: reread staged **local** histogram/source files and rebuild
  their contexts within every learned request. Staging itself is recorded initialization. Neither mode guarantees
  cold OS/storage caches. Keep these cache settings with any reported speedup.
- Feature code is reused from the repository; a small embedded Java adapter calls existing histogram, feature,
  LightGBM and exact-counter classes. It separates timing stages without changing histogram algorithms.
  Java currently computes source statistics in blocks: A already shares most source work with B.

## Correctness before speed

Rebuild the full saved cohort, verify input/model identities, predict with saved weights and reconstruct every
selected policy's full-test counts. Convert finite saved log cutoffs to the Java error-score scale; retain
negative infinity as strict reject-all. Then check Java score and acceptance agreement on every timed case.
Any decision mismatch stops the run; never repair it by tuning on test truth.

Sampling uses only saved test IDs and a deterministic hash, shared across rates/methods. Calibration queries provide explicit warm-up; a prior test-case verification pass also warms the runtime. Exact counting uses inclusive rectangle bounds and counts duplicate points. Supplied truth is used
only for validation/scoring; it never substitutes for a scan. Primary eligibility stays `truth >= 10`, as in 08d.
If an expanded cohort contains zero truth, error is defined as 0 for answer 0 and infinity otherwise; this notebook
does not add those cases to the original saved cohort.

## Input and identity checks

In [ ]:
def read_csv(path):
    return pd.read_csv(path, float_precision='round_trip')


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def safe_path(value):
    value = str(value)
    if any(char in value for char in ',\"\r\n'):
        raise ValueError('Paths must not contain commas, quotes or line breaks: ' + value)
    return Path(value).expanduser().resolve()


def bool_series(series):
    converted = series.astype(str).str.lower().map({'true': True, 'false': False})
    if converted.isna().any():
        raise ValueError('Expected explicit True/False values')
    return converted.astype(bool)


def canonical_rates(values, available):
    result = []
    for value in values:
        matched = [r for r in available if math.isclose(float(value), r, rel_tol=0, abs_tol=1e-12)]
        if len(matched) != 1:
            raise ValueError('Unknown or ambiguous saved merge rate: ' + str(value))
        result.append(matched[0])
    return result


def resolve_inputs(source, prior, cfg):
    expected = prior.get('input_sha256', {})
    if not expected:
        raise ValueError('08d run.json is missing input SHA256 identities')
    found = {}
    for key, digest in expected.items():
        group, filename = key.split('/', 1)
        override = cfg['data_dir'] if group == 'data' else cfg['histogram_dir']
        candidates = [safe_path(override) / filename] if override else [source / 'inputs' / group / filename]
        original_root = prior['config']['data_root'] if group == 'data' else prior['config']['sample_root']
        if not override:
            candidates.append(Path(original_root) / prior['config']['dataset'] / filename)
        path = next((p for p in candidates if p.is_file()), None)
        if path is None:
            raise FileNotFoundError('Missing 08d input ' + key + '; set data_dir/histogram_dir to matching inputs')
        if file_hash(path) != digest:
            raise ValueError('Input SHA256 differs from 08d: ' + str(path))
        found[key] = path.resolve()
    for required in ('data/queries-Ans.txt', 'data/queries-meta.csv', 'run/rows.csv'):
        if required not in found:
            raise ValueError('Missing required input identity: ' + required)
    return found


def choose_query_ids(ids, limit, seed):
    # No label, score, merge-rate or model information participates in sampling.
    ordered = sorted(map(int, ids), key=lambda q: hashlib.sha256(f'{seed}:{q}'.encode()).digest())
    return sorted(ordered if limit == 0 else ordered[:limit])


def find_math_jar(cfg):
    if cfg['math_jar']:
        path = safe_path(cfg['math_jar'])
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    roots = [Path('/databricks/jars'), Path.home() / '.m2/repository/org/apache/commons/commons-math3',
             safe_path(cfg['jar_path']).parent, Path('/workspace/.synopsislake/m2/org/apache/commons/commons-math3')]
    for root in roots:
        if root.exists():
            matches = sorted(root.rglob('*commons-math3*.jar'))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError('Set math_jar to commons-math3-3.6.1.jar; no automatic download/deployment is performed')


def load_configuration():
    cfg = {key: str(get_setting(key)).strip() for key in DEFAULTS}
    if not cfg['run_dir']:
        raise ValueError('Set run_dir to the COMPLETED 08d run directory containing run.json and models/. '
                         'The compact results ZIP does not contain model weights.')
    for key in ('max_queries', 'repeats', 'warmup_queries', 'num_threads', 'random_seed'):
        cfg[key] = int(cfg[key])
    if cfg['max_queries'] < 0 or cfg['repeats'] < 1 or cfg['warmup_queries'] < 0 or cfg['num_threads'] < 1:
        raise ValueError('Invalid query limit, repetition, warm-up or thread count')
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    cfg['policies'] = cfg['policies'].split(',')
    cfg['cache_modes'] = cfg['cache_modes'].split(',')
    for key in ('seeds', 'policies', 'cache_modes'):
        if not cfg[key] or len(set(cfg[key])) != len(cfg[key]):
            raise ValueError('Empty or duplicate setting: ' + key)
    if not set(cfg['policies']) <= {'empirical', 'grid_empirical', 'ucb_grid'}:
        raise ValueError('Use saved empirical, grid_empirical or ucb_grid policies')
    if not set(cfg['cache_modes']) <= {'resident', 'reload'}:
        raise ValueError('cache_modes must contain resident and/or reload')
    return cfg

## Prepare frozen models, cohorts and reference decisions (outside request timing)

In [ ]:
def prepare_benchmark(cfg, destination, state):
    source, repo = safe_path(cfg['run_dir']), safe_path(cfg['repo_root'])
    prior = json.loads((source / 'run.json').read_text())
    if prior.get('status') != 'complete' or prior.get('study') != 'source-information':
        raise ValueError('Requires a completed 08d source-information run')
    if float(prior['config']['delta']) != 0.05:
        raise ValueError('This benchmark preserves the 08d delta=0.05 setting')
    epsilon, delta = 0.25, 0.05
    if epsilon not in [float(x) for x in str(prior['config']['epsilons']).split(',')]:
        raise ValueError('The source run has no saved epsilon=0.25 evaluation')
    for name in ('features.py', 'features_v1.py'):
        if file_hash(repo / 'ml' / name) != prior.get('code_sha256', {}).get(name):
            raise ValueError('Feature definition code differs from 08d: ' + name)
    sys.path.insert(0, str(repo / 'ml'))
    import features
    import features_v1
    for module in (features, features_v1):
        if Path(module.__file__).resolve().parent != repo / 'ml':
            raise ValueError('A different repository feature module is already imported; restart Python')
    importlib.invalidate_caches()
    importlib.reload(features)
    importlib.reload(features_v1)
    inputs = resolve_inputs(source, prior, cfg)
    # Stage small saved artifacts for the standalone Java process. Staging is setup, not request I/O.
    # Reload mode subsequently reads these local staged histogram files, not remote physical storage.
    stage = Path(tempfile.mkdtemp(prefix='synopsislake-e2e-'))
    state['staging_directory'] = str(stage)
    for group in ('data', 'run', 'models'):
        (stage / group).mkdir()
    tick = time.perf_counter()
    for key, path in inputs.items():
        # rows.csv is needed by Python verification, but need not be copied for Java.
        if key != 'run/rows.csv':
            shutil.copyfile(path, stage / key)
            if file_hash(stage / key) != prior['input_sha256'][key]:
                raise ValueError('Input changed during staging: ' + key)
    state['setup_seconds']['artifact_staging'] = time.perf_counter() - tick
    tick = time.perf_counter()
    frame = features_v1.build_table_v1(str(stage / 'run'), str(stage / 'data'), rows_csv=str(inputs['run/rows.csv']))
    features.check_estimates(frame)
    features_v1.check_source_estimates(frame)
    if frame.empty or frame.duplicated(['qId', 'mergeRate']).any():
        raise ValueError('Empty feature cohort or duplicate query/rate pairs')
    available = sorted(map(float, frame.mergeRate.unique()))
    if cfg['merge_rates'].lower() == 'all':
        rates = available
    else:
        requested = [float(x) for x in cfg['merge_rates'].split(',')]
        rates = sorted({min(available, key=lambda r: abs(r - x)) for x in requested})
    cfg['resolved_merge_rates'] = rates
    state['setup_seconds']['offline_feature_verification'] = time.perf_counter() - tick
    if file_hash(inputs['run/rows.csv']) != prior['input_sha256']['run/rows.csv']:
        raise ValueError('rows.csv changed during reference verification')
    queries = pd.read_csv(
        stage / 'data/queries-Ans.txt', header=None, float_precision='round_trip')
    if queries.shape[1] != 5 or not np.isfinite(queries.to_numpy(float)).all():
        raise ValueError('queries-Ans.txt must contain finite bounds and supplied truth in five columns')
    queries.columns = ['minLon', 'maxLon', 'minLat', 'maxLat', 'truth']
    if ((queries.minLon >= queries.maxLon) | (queries.minLat >= queries.maxLat)).any():
        raise ValueError('Invalid query rectangle')
    if not np.array_equal(frame['gt'].to_numpy(), queries.loc[frame.qId, 'truth'].to_numpy()):
        raise ValueError('rows.csv truth does not match query metadata')
    arm_paths = {}
    for path in (stage / 'run').glob('arm_*.txt'):
        rate = canonical_rates([float(path.read_text().splitlines()[0])], available)[0]
        if rate in arm_paths:
            raise ValueError('Duplicate ARM histogram rate')
        arm_paths[rate] = str(path)
    if set(arm_paths) != set(available):
        raise ValueError('Missing ARM histogram for an eligible rate')
    splits = read_csv(source / 'splits.csv')
    if splits.duplicated(['seed', 'qId']).any():
        raise ValueError('Duplicate split IDs')
    gates = read_csv(source / 'gate_details.csv')
    budgets = sorted(set(gates.tree_budget.astype(int)))
    budget = int(cfg['tree_budget']) if cfg['tree_budget'] else budgets[0] if len(budgets) == 1 else None
    if budget is None:
        raise ValueError('Choose an explicit tree_budget from the saved run')
    cfg['resolved_tree_budget'] = budget
    base = [name for name in features.FEATURES if name not in {'skew_pred', 'log_alpha'}]
    definitions = {VARIANTS[0]: base + ['src_edge_contrast'], VARIANTS[1]: base + list(features_v1.SOURCE_FEATURES)}
    if [len(definitions[v]) for v in VARIANTS] != [18, 24]:
        raise ValueError('Unexpected feature definitions')
    model_rows, gate_rows, audits, case_rows, warm_rows, reference_rows = [], [], [], [], [], []
    tick = time.perf_counter()
    for seed in cfg['seeds']:
        subset = splits[splits.seed == seed].copy()
        if set(subset.qId) != set(frame.qId) or set(subset.split) != {'train', 'calibration', 'test'}:
            raise ValueError('Saved split does not cover the exact eligible cohort for seed ' + str(seed))
        test_ids = sorted(subset.loc[subset.split == 'test', 'qId'].astype(int))
        cal_ids = sorted(subset.loc[subset.split == 'calibration', 'qId'].astype(int))
        sample_ids = choose_query_ids(test_ids, cfg['max_queries'], cfg['random_seed'] + seed)
        warm_ids = choose_query_ids(cal_ids, cfg['warmup_queries'], cfg['random_seed'] + seed) if cfg['warmup_queries'] else []
        expected_test = frame[frame.qId.isin(test_ids)].sort_values(['qId', 'mergeRate']).reset_index(drop=True)
        scores = {}
        for variant in VARIANTS:
            folder = source / 'models' / variant / f'trees_{budget}' / f'seed_{seed}'
            schema = json.loads((folder / 'schema.json').read_text())
            if (schema.get('features') != definitions[variant] or prior['feature_definitions'][variant] != definitions[variant]
                    or schema.get('objective') != 'regression' or schema.get('score') != 'predicted log-error'
                    or schema.get('min_exact') != 10 or schema.get('seed') != seed or schema.get('tree_budget') != budget):
                raise ValueError('Incompatible saved model schema: ' + str(folder))
            origins = [row for row in prior['model_sources'] if row['variant'] == variant
                       and row['seed'] == seed and row['tree_budget'] == budget]
            digest = file_hash(folder / 'model.txt')
            if len(origins) != 1 or origins[0]['model_sha256'] != digest:
                raise ValueError('Saved model identity mismatch')
            model_path = stage / 'models' / f'{variant}-{seed}.txt'
            shutil.copyfile(folder / 'model.txt', model_path)
            if file_hash(model_path) != digest:
                raise ValueError('Model changed during staging')
            booster = lgb.Booster(model_file=str(model_path))
            if booster.feature_name() != definitions[variant] or booster.params.get('objective') != 'regression':
                raise ValueError('Model feature order/objective differs from schema')
            cohorts = read_csv(folder / 'cohorts.csv')
            cohorts['mergeRate'] = canonical_rates(cohorts.mergeRate, available)
            expected_cohorts = frame[['qId', 'mergeRate']].merge(subset[['qId', 'split']], on='qId', validate='many_to_one')
            if cohorts.duplicated(['qId', 'mergeRate']).any():
                raise ValueError('Duplicate saved cohort rows')
            pd.testing.assert_frame_equal(cohorts[['qId', 'mergeRate', 'split']].sort_values(['qId', 'mergeRate']).reset_index(drop=True),
                expected_cohorts.sort_values(['qId', 'mergeRate']).reset_index(drop=True), check_dtype=False)
            if not np.isfinite(expected_test[definitions[variant]].to_numpy(float)).all():
                raise ValueError('Nonfinite model features')
            prediction = expected_test[['qId', 'mergeRate', 'relErr']].copy()
            prediction['score'] = booster.predict(expected_test[definitions[variant]], num_threads=cfg['num_threads'])
            if not np.isfinite(prediction.score).all():
                raise ValueError('Nonfinite prediction')
            saved = read_csv(folder / 'test_predictions.csv')
            if saved.duplicated(['qId', 'mergeRate']).any():
                raise ValueError('Duplicate saved predictions')
            saved['mergeRate'] = canonical_rates(saved.mergeRate, available)
            saved = saved.sort_values(['qId', 'mergeRate']).reset_index(drop=True)
            if not prediction[['qId', 'mergeRate']].equals(saved[['qId', 'mergeRate']]):
                raise ValueError('Saved test prediction cohort differs')
            if not np.allclose(prediction.score, saved.score, rtol=1e-8, atol=1e-10):
                raise ValueError('Current Python scores differ from saved 08d predictions')
            if not np.allclose(prediction.relErr, saved.relErr, rtol=1e-10, atol=1e-12):
                raise ValueError('Saved test errors differ from original feature table')
            scores[variant] = prediction.set_index(['qId', 'mergeRate']).score
            model_rows.append(dict(seed=seed, variant=variant, modelPath=str(model_path)))
            state['models'].append(dict(variant=variant, seed=seed, sha256=digest,
                features=definitions[variant], original_08d_fit_seconds=origins[0]['current_fit_seconds'],
                earlier_source_fit_seconds=origins[0].get('source_fit_seconds'),
                original_calibration_seconds=None,
                original_calibration_note='08d did not record calibration alone; evaluation includes calibration and diagnostics',
                original_evaluation_seconds=float(read_csv(folder / 'results.csv').evaluation_seconds.iloc[0])
                    if (folder / 'results.csv').exists() else None))
            for policy in cfg['policies']:
                selected = gates[(gates.variant == variant) & (gates.seed == seed) & (gates.tree_budget == budget)
                    & (gates.score == 'learned') & (gates.policy == policy) & (gates.scope == 'per_rate')
                    & np.isclose(gates.eps, epsilon, rtol=0, atol=1e-12)].copy()
                selected['mergeRate'] = canonical_rates(selected.mergeRate, available)
                if selected.mergeRate.duplicated().any() or set(selected.mergeRate) != set(available):
                    raise ValueError('Missing or duplicate frozen gate rows')
                accept_none = bool_series(selected.accept_none)
                for (_, gate), none in zip(selected.iterrows(), accept_none):
                    rate = gate.mergeRate
                    cut = -math.inf if none else float(gate.cutoff)
                    if not none and not math.isfinite(cut):
                        raise ValueError('Invalid saved finite gate cutoff')
                    p = prediction[prediction.mergeRate == rate]
                    accepted = p.score <= cut
                    saved_accepted = saved.loc[saved.mergeRate == rate, 'score'].to_numpy() <= cut
                    if not np.array_equal(accepted.to_numpy(), saved_accepted):
                        raise ValueError('Python acceptance decisions changed from saved predictions')
                    n, bad = int(accepted.sum()), int((accepted & (p.relErr > epsilon)).sum())
                    if (len(p), n, bad) != (int(gate.test_count), int(gate.accepted_count), int(gate.violation_count)):
                        raise ValueError('Frozen gate counts disagree with gate_details.csv')
                    audits.append(dict(seed=seed, variant=variant, policy=policy, mergeRate=rate, test_count=len(p),
                        accepted_count=n, violation_count=bad, acceptance=n / len(p), violation=bad / n if n else np.nan,
                        matches_saved=True))
                    if rate in rates:
                        gate_rows.append(dict(seed=seed, variant=variant, policy=policy, mergeRate=rate, cutoff_log=cut))
            prediction.assign(seed=seed, variant=variant).to_csv(destination / f'reference-{variant}-{seed}.csv', index=False)
        for rate in rates:
            for ids, target in ((sample_ids, case_rows), (warm_ids, warm_rows)):
                for qid in ids:
                    q = queries.loc[qid]
                    # Warm-up queries come from calibration; their labels never select a gate.
                    expected_a = scores[VARIANTS[0]].loc[(qid, rate)] if target is case_rows else ''
                    expected_b = scores[VARIANTS[1]].loc[(qid, rate)] if target is case_rows else ''
                    target.append(dict(seed=seed, qId=qid, mergeRate=rate, armPath=arm_paths[rate],
                        **q.to_dict(), expectedAlog=expected_a, expectedBlog=expected_b,
                        expectedEstimate=float(expected_test.loc[(expected_test.qId == qid) & (expected_test.mergeRate == rate), 'est'].iloc[0])
                        if target is case_rows else ''))
    state['setup_seconds']['offline_model_and_gate_verification'] = time.perf_counter() - tick
    pd.DataFrame(audits).to_csv(destination / 'gate_audit.csv', index=False)
    pd.DataFrame(case_rows).to_csv(destination / 'selected_queries.csv', index=False)
    pd.DataFrame(warm_rows, columns=pd.DataFrame(case_rows).columns).to_csv(stage / 'warmup.csv', index=False)
    pd.DataFrame(case_rows).to_csv(stage / 'cases.csv', index=False)
    pd.DataFrame(model_rows).to_csv(stage / 'models.csv', index=False)
    for policy in cfg['policies']:
        pd.DataFrame(gate_rows).query('policy == @policy').drop(columns='policy').to_csv(stage / f'gates-{policy}.csv', index=False)
    sources = sorted((stage / 'run').glob('src_*.txt'))
    if not sources:
        raise ValueError('Missing source histograms')
    (stage / 'sources.txt').write_text(''.join(str(p) + '\n' for p in sources))
    raw_paths = [safe_path(cfg['raw_dir']) / name.strip() for name in cfg['raw_files'].split(',')]
    if len(set(raw_paths)) != len(raw_paths) or not raw_paths:
        raise ValueError('Missing or duplicate raw input files')
    tick = time.perf_counter()
    state['raw_files'] = []
    for path in raw_paths:
        before = path.stat()
        digest = file_hash(path)
        after = path.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise ValueError('Raw input changed while hashing')
        state['raw_files'].append(dict(path=str(path), size=after.st_size, mtime_ns=after.st_mtime_ns, sha256=digest))
    state['setup_seconds']['raw_identity_hashing'] = time.perf_counter() - tick
    try:
        state['git_commit'] = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
        state['git_dirty'] = bool(subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, stderr=subprocess.DEVNULL))
    except (OSError, subprocess.CalledProcessError):
        state['git_commit'] = None
    notebook = repo / 'ml/notebooks/09_e2e_source_information.ipynb'
    state['notebook_sha256'] = file_hash(notebook) if notebook.exists() else None
    (stage / 'raw_files.txt').write_text(''.join(str(p) + '\n' for p in raw_paths))
    state.update(source_run=str(source), source_run_sha256=file_hash(source / 'run.json'),
        input_sha256=prior['input_sha256'], feature_code_sha256={n: file_hash(repo / 'ml' / n) for n in ('features.py', 'features_v1.py')},
        source_08d_timings=prior.get('timings', {}), epsilon=epsilon, delta=delta,
        current_training_seconds=0.0, current_calibration_seconds=0.0,
        selected_query_cases=len(case_rows), selected_unique_qids=len({r['qId'] for r in case_rows}))
    return stage

## Compile and execute the existing-engine adapter

In [ ]:
def execute_java(cfg, destination, state, stage):
    jar, math_jar = safe_path(cfg['jar_path']), find_math_jar(cfg)
    if not jar.is_file():
        raise FileNotFoundError('Build/upload a JAR from current team main: ' + str(jar))
    required = {'predictor/LightGBMModel.class', 'predictor/FeatureRegistry.class', 'predictor/QueryFeatures.class',
                'predictor/PointScanCounter.class', 'predictor/SynopsisContext.class'}
    with ZipFile(jar) as archive:
        if not required <= set(archive.namelist()):
            raise ValueError('The supplied JAR lacks the learned query runtime; rebuild current team main')
    state['jar_sha256'], state['math_jar_sha256'] = file_hash(jar), file_hash(math_jar)
    java_source = stage / 'SourceInformationBenchmark.java'
    java_source.write_text(JAVA_SOURCE)
    state['adapter_sha256'] = file_hash(java_source)
    compile_cp = os.pathsep.join((str(jar), str(math_jar)))
    compile_cmd = [cfg['javac'], '-encoding', 'UTF-8', '-source', '8', '-target', '8', '-cp', compile_cp, str(java_source)]
    tick = time.perf_counter()
    compiled = subprocess.run(compile_cmd, capture_output=True, text=True)
    (destination / 'compile.log').write_text(compiled.stdout + compiled.stderr)
    if compiled.returncode:
        raise RuntimeError('Adapter compilation failed; inspect compile.log. A classic dedicated cluster with a JDK is required.')
    state['setup_seconds']['adapter_compilation'] = time.perf_counter() - tick
    cp = os.pathsep.join((str(stage), compile_cp))
    state['java_version'] = subprocess.run([cfg['java'], '-version'], capture_output=True, text=True, check=True).stderr.strip()
    for policy in cfg['policies']:
        out = destination / policy
        out.mkdir()
        props = dict(cases=stage / 'cases.csv', warmup=stage / 'warmup.csv', models=stage / 'models.csv',
                     gates=stage / f'gates-{policy}.csv', sources=stage / 'sources.txt', raw_files=stage / 'raw_files.txt',
                     output=out, repeats=cfg['repeats'], cache_modes=','.join(cfg['cache_modes']), random_seed=cfg['random_seed'])
        config_path = stage / f'benchmark-{policy}.properties'
        config_path.write_text(''.join(f'{k}={v}\n' for k, v in props.items()))
        command = [cfg['java'], '-cp', cp, 'SourceInformationBenchmark', str(config_path)]
        (out / 'command.json').write_text(json.dumps(command, indent=2))
        tick = time.perf_counter()
        with (out / 'execution.log').open('w') as log:
            process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
            print('Running', policy, '| log:', out / 'execution.log', flush=True)
            try:
                while process.poll() is None:
                    time.sleep(1)
                    if time.perf_counter() - tick > 30:
                        # Keep progress visible during long Databricks runs without flooding notebook output.
                        elapsed = int(time.perf_counter() - tick)
                        if elapsed % 30 == 0:
                            print(policy, 'elapsed seconds:', elapsed, flush=True)
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=10)
                raise
            if process.returncode:
                raise RuntimeError('Java validation/execution failed: ' + (out / 'execution.log').read_text()[-5000:])
        state['setup_seconds'][f'java_process_wall_{policy}'] = time.perf_counter() - tick
        print('Completed:', out, flush=True)
    for item in state['raw_files']:
        stat = Path(item['path']).stat()
        if (stat.st_size, stat.st_mtime_ns) != (item['size'], item['mtime_ns']):
            raise ValueError('Raw file changed during benchmark; results are invalid')

## Summarize repetitions without duplicating answer-quality cases

In [ ]:
def summarize_benchmark(destination, cfg, state):
    frames = []
    for policy in cfg['policies']:
        rows = read_csv(destination / policy / 'measurements.csv')
        rows['policy'] = policy
        rows['accepted'] = bool_series(rows.accepted)
        frames.append(rows)
    measurements = pd.concat(frames, ignore_index=True)
    keys = ['policy', 'cache_mode', 'seed', 'mergeRate', 'qId', 'method']
    if measurements.duplicated(keys + ['repeat']).any():
        raise ValueError('Duplicate timing observations')
    repeated = measurements.groupby(keys, dropna=False)
    if not (repeated.size() == cfg['repeats']).all():
        raise ValueError('Incomplete repetitions')
    if (repeated[['accepted', 'answer', 'truth']].nunique() > 1).any().any():
        raise ValueError('Answers or decisions changed across repetitions')
    unique = measurements.drop_duplicates(keys).copy()
    unique['relative_error'] = np.where(unique.truth > 0, (unique.answer - unique.truth).abs() / unique.truth,
                                        np.where(unique.answer == 0, 0.0, np.inf))
    unique['violating_answer'] = unique.relative_error > state['epsilon']
    unique['accepted_violation'] = unique.accepted & unique.violating_answer
    unique['good_accepted'] = unique.accepted & ~unique.violating_answer
    measurements = measurements.merge(unique[keys + ['relative_error', 'violating_answer', 'accepted_violation']], on=keys,
                                      validate='many_to_one')
    measurements.to_csv(destination / 'per_query.csv', index=False)
    unique.to_csv(destination / 'quality_cases.csv', index=False)
    group_keys = ['policy', 'cache_mode', 'seed', 'mergeRate', 'method']
    stages = ['histogram_load_ns', 'estimate_ns', 'features_ns', 'inference_ns', 'decision_ns', 'exact_ns', 'total_ns']
    summary_rows = []
    for values, group in measurements.groupby(group_keys, sort=True):
        info = dict(zip(group_keys, values))
        quality = unique
        for key, value in info.items():
            quality = quality[quality[key] == value]
        n, accepted = len(quality), int(quality.accepted.sum())
        bad = int(quality.accepted_violation.sum())
        row = dict(**info, query_count=n, timing_observations=len(group), repeats=cfg['repeats'],
                   accepted_count=accepted, violation_count=bad, good_accepted_count=accepted - bad,
                   acceptance=accepted / n, accepted_violation_rate=bad / accepted if accepted else np.nan,
                   quality_target_met=(bad / accepted <= state['delta']) if accepted else None,
                   returned_violation_count=int(quality.violating_answer.sum()),
                   raw_access='resident arrays', histogram_access='resident' if info['cache_mode'] == 'resident' else 'local staged reload',
                   model_access='resident', exploratory=True)
        for column in stages:
            row[column.replace('_ns', '_mean_ms')] = float(group[column].mean()) / 1e6
            row[column.replace('_ns', '_p95_ms')] = float(group[column].quantile(0.95)) / 1e6
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    base_keys = ['policy', 'cache_mode', 'seed', 'mergeRate']
    baseline = summary[summary.method == 'always_exact'][base_keys + ['total_mean_ms', 'total_p95_ms']].rename(
        columns={'total_mean_ms': 'baseline_total_mean_ms', 'total_p95_ms': 'baseline_total_p95_ms'})
    summary = summary.merge(baseline, on=base_keys, validate='many_to_one')
    summary['speedup_mean'] = summary.baseline_total_mean_ms / summary.total_mean_ms
    summary['speedup_p95_ratio'] = summary.baseline_total_p95_ms / summary.total_p95_ms
    summary.to_csv(destination / 'summary.csv', index=False)
    # Retain request pairs; do not turn repeats or cross-rate rows into independent samples.
    pair_keys = base_keys + ['qId', 'repeat']
    pairs = measurements.pivot(index=pair_keys, columns='method', values='total_ns').reset_index()
    if pairs[['always_exact', *VARIANTS]].isna().any().any():
        raise ValueError('Incomplete method pairs')
    pairs.to_csv(destination / 'paired_timings.csv', index=False)
    setups = []
    for policy in cfg['policies']:
        s = read_csv(destination / policy / 'setup.csv')
        s['policy'] = policy
        setups.append(s)
    pd.concat(setups, ignore_index=True).to_csv(destination / 'setup.csv', index=False)
    interpretation = (
        '# E2E source-information benchmark\n\n'
        'The saved 08d A18/B24 weights, cohorts and calibration thresholds were reused. No model was trained '
        'and no cutoff was recalibrated. `gate_audit.csv` verifies full saved test-cohort counts outside timing; '
        '`summary.csv` reports only the deterministically selected timed cases.\n\n'
        'Compare latency and accepted-answer violation together. Zero accepted answers have undefined risk. '
        'Each summary row is one seed, merge rate, policy, cache mode and method. Repetitions affect latency only; '
        'query IDs repeat across rates/seeds, so these are not independent statistical trials. '
        'The p95 ratio is a ratio of marginal latency quantiles, not a percentile of paired speedups.\n\n'
        'Raw points and models are resident in every timed mode. Resident mode also preloads histogram contexts. '
        'Reload mode includes reading staged local histogram/source files and creating their contexts on each learned request. '
        'It is not cold remote-storage execution. Artifact staging, hashing, Python reference validation, raw loading, '
        'model loading, resident histogram preprocessing, parity checking and warm-up are initialization/verification costs; '
        'consult run.json and setup.csv. The test-case parity and raw-truth verification prepass also warms the runtime before explicit calibration warm-up. Process wall time includes initialization, validation and output overhead and is not '
        'per-request latency. Original 08d fit/evaluation times are metadata, not current query costs.\n\n'
        'The current Java implementation computes source-statistic blocks together even for A. These timings compare '
        'the actual implementation, not an optimized contrast-only feature extractor. Fresh holdout queries are still '
        'needed for a final quality claim; previously inspected 08d test queries make this comparison exploratory.\n')
    (destination / 'analysis_en.md').write_text(interpretation)
    return summary


def export_results(destination):
    with ZipFile(destination / 'e2e_results.zip', 'w', ZIP_DEFLATED) as archive:
        for path in destination.rglob('*'):
            if path.is_file() and path.suffix in {'.csv', '.json', '.md', '.log', '.png'}:
                archive.write(path, path.relative_to(destination))


def run_benchmark(cfg):
    destination = safe_path(cfg['output_root']) / ('e2e-source-information-' + uuid4().hex[:12])
    destination.mkdir(parents=True, exist_ok=False)
    global _benchmark_last_result_dir
    _benchmark_last_result_dir = str(destination)
    state = dict(status='running', study='e2e-source-information', started_utc=datetime.now(timezone.utc).isoformat(),
                 config=dict(cfg), setup_seconds={}, models=[], python=platform.python_version(),
                 package_versions={name: importlib.metadata.version(name) for name in ('lightgbm', 'numpy', 'pandas', 'scipy')},
                 cache_note='Raw/model resident; reload means staged local histogram/source reload; OS cache uncontrolled; full selected-test verification precedes calibration warm-up')
    started = time.perf_counter()
    write_json(destination / 'run.json', state)
    try:
        tick = time.perf_counter()
        stage = prepare_benchmark(cfg, destination, state)
        state['setup_seconds']['preparation_wall'] = time.perf_counter() - tick
        state['config'] = dict(cfg)
        write_json(destination / 'run.json', state)
        execute_java(cfg, destination, state, stage)
        summary = summarize_benchmark(destination, cfg, state)
        state.update(status='complete', experiment_wall_seconds=time.perf_counter() - started,
                     finished_utc=datetime.now(timezone.utc).isoformat())
        write_json(destination / 'run.json', state)
        export_results(destination)
        print('COMPLETE:', destination)
        return destination, summary
    except BaseException as error:
        state.update(status='failed', error=f'{type(error).__name__}: {error}',
                     experiment_wall_seconds=time.perf_counter() - started)
        write_json(destination / 'run.json', state)
        print('FAILED; diagnostic files preserved:', destination)
        raise

## Java runtime adapter

The source below is written to a temporary directory and compiled against the supplied JAR.
No repository Java source is patched. The saved models are regression models predicting log relative error.

In [ ]:
JAVA_SOURCE = r"""
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import histogram.SimpleSpatialHistogramOpt;
import predictor.*;

/** Runtime adapter only: no training, threshold fitting, or cached answers. */
public final class SourceInformationBenchmark {
    static final String A="no_sa_contrast", B="no_sa_all_source";
    static final String[] METHODS={"always_exact",A,B};
    static Properties config=new Properties();
    static List<String> sources;
    static PointScanCounter raw;
    static PrintWriter setup;
    static Map<String,Model> models=new LinkedHashMap<>();
    static Map<String,Double> gates=new LinkedHashMap<>();
    static Map<String,Hist> resident=new LinkedHashMap<>();
    static class Model {
        LightGBMModel booster; FeatureRegistry.Feature[] features;
        Model(String path,String variant) throws Exception {
            booster=LightGBMModel.load(path);
            List<String> names=Arrays.asList(booster.getFeatureNames());
            if(names.contains("skew_pred")||names.contains("log_alpha")) throw new IllegalArgumentException("SA input in "+path);
            if(names.size()!=(variant.equals(A)?18:24)) throw new IllegalArgumentException("Wrong feature count in "+path);
            features=FeatureRegistry.resolve(booster.getFeatureNames());
        }
    }
    static class Hist { SimpleSpatialHistogramOpt histogram; SynopsisContext context; }
    static class Query {
        int seed,id; double rate,x0,x1,y0,y1,truth,a,b,expectedEstimate; String arm;
        Query(Map<String,String> row) {
            seed=Integer.parseInt(row.get("seed")); id=Integer.parseInt(row.get("qId"));
            rate=number(row.get("mergeRate")); arm=row.get("armPath");
            x0=number(row.get("minLon")); x1=number(row.get("maxLon"));
            y0=number(row.get("minLat")); y1=number(row.get("maxLat")); truth=number(row.get("truth"));
            a=optional(row.get("expectedAlog")); b=optional(row.get("expectedBlog")); expectedEstimate=optional(row.get("expectedEstimate"));
            if(!Double.isFinite(rate)||!Double.isFinite(truth)||truth<0||!Double.isFinite(x0)||!Double.isFinite(x1)||!Double.isFinite(y0)||!Double.isFinite(y1)||x0>x1||y0>y1)
                throw new IllegalArgumentException("Invalid query "+id);
        }
        double expected(String v) {return v.equals(A)?a:b;}
        String rateKey() {return seed+":"+Double.toString(rate);}
        String histKey() {return arm+":"+Double.toString(rate);}
    }
    static class Result {
        long load,estimate,features,inference,decision,exact,total;
        boolean accepted; double answer,log=Double.NaN,error=Double.NaN,cutLog=Double.NaN,cutError=Double.NaN;
    }
    static double number(String s) {
        if(s.equalsIgnoreCase("-inf")) return Double.NEGATIVE_INFINITY;
        if(s.equalsIgnoreCase("inf")||s.equalsIgnoreCase("+inf")) return Double.POSITIVE_INFINITY;
        return Double.parseDouble(s);
    }
    static double optional(String s) {return s==null||s.isEmpty()?Double.NaN:number(s);}
    static String required(String key) {String v=config.getProperty(key); if(v==null||v.trim().isEmpty()) throw new IllegalArgumentException("Missing property "+key); return v;}
    static List<String> paths(String file) throws Exception {
        List<String> out=new ArrayList<>();
        for(String s:Files.readAllLines(Paths.get(file),StandardCharsets.UTF_8)) if(!s.trim().isEmpty()) out.add(s.trim());
        if(out.isEmpty()) throw new IllegalArgumentException("Empty path list "+file); return out;
    }
    static List<Map<String,String>> csv(String file) throws Exception {
        List<String> lines=Files.readAllLines(Paths.get(file),StandardCharsets.UTF_8);
        if(lines.isEmpty()) throw new IllegalArgumentException("Empty CSV "+file);
        // Paths containing commas, quotes or line breaks are rejected by notebook preflight.
        String[] head=lines.get(0).split(",",-1); List<Map<String,String>> rows=new ArrayList<>();
        for(int i=1;i<lines.size();i++) {
            if(lines.get(i).isEmpty()) continue;
            String[] values=lines.get(i).split(",",-1);
            if(values.length!=head.length) throw new IllegalArgumentException("CSV field count at "+file+":"+(i+1));
            Map<String,String> row=new LinkedHashMap<>(); for(int j=0;j<head.length;j++) row.put(head[j],values[j]); rows.add(row);
        } return rows;
    }
    static PointScanCounter loadRaw(List<String> files) throws Exception {
        PointScanCounter counter=new PointScanCounter();
        for(String file:files) try(BufferedReader reader=Files.newBufferedReader(Paths.get(file),StandardCharsets.UTF_8)) {
            String line; int row=0;
            while((line=reader.readLine())!=null) {
                row++; String[] fields=line.split("\\t",-1);
                if(fields.length<2) throw new IllegalArgumentException("Raw row needs tab-separated lon and lat: "+file+":"+row);
                double x=number(fields[0]),y=number(fields[1]);
                if(!Double.isFinite(x)||!Double.isFinite(y)) throw new IllegalArgumentException("Nonfinite raw coordinate: "+file+":"+row);
                counter.add(x,y);
            }
        }
        return counter;
    }
    static List<Query> queries(String file) throws Exception {
        List<Query> out=new ArrayList<>(); Set<String> seen=new HashSet<>();
        for(Map<String,String> r:csv(file)) {Query q=new Query(r); if(!seen.add(q.rateKey()+":"+q.id)) throw new IllegalArgumentException("Duplicate query key "+q.id); out.add(q);} return out;
    }
    static void setupRow(String stage,int seed,String variant,double rate,String cache,long elapsed,String detail) {
        setup.println(stage+","+seed+","+variant+","+rate+","+cache+","+elapsed+","+detail.replace(',',';').replace('\n',' ')); setup.flush();
    }
    static Hist load(Query q) throws Exception {
        HistogramIO.Loaded merged=HistogramIO.read(q.arm);
        if(Math.abs(merged.mergeRate-q.rate)>1e-10) throw new IllegalArgumentException("Histogram merge rate mismatch "+q.arm);
        SimpleSpatialHistogramOpt[] src=new SimpleSpatialHistogramOpt[sources.size()];
        for(int i=0;i<src.length;i++) src[i]=HistogramIO.read(sources.get(i)).hist;
        Hist h=new Hist(); h.histogram=merged.hist; h.context=new SynopsisContext(h.histogram,src,q.rate,null); return h;
    }
    static double[] values(Model m,Hist h,Query q) {
        QueryFeatures f=new QueryFeatures(h.context,q.x0,q.x1,q.y0,q.y1); double[] x=new double[m.features.length];
        for(int i=0;i<x.length;i++) x[i]=m.features[i].value(f); return x;
    }
    static double cut(Query q,String method) {
        Double v=gates.get(q.rateKey()+":"+method); if(v==null||Double.isNaN(v)||v==Double.POSITIVE_INFINITY) throw new IllegalArgumentException("Missing or invalid gate "+q.rateKey()+":"+method); return v;
    }
    static double transformedCut(double log) {return log==Double.NEGATIVE_INFINITY?Double.NEGATIVE_INFINITY:Math.exp(log);}
    static Result execute(Query q,String method,String cache) throws Exception {
        long start=System.nanoTime(); Result r=new Result();
        if(method.equals("always_exact")) {
            long t=System.nanoTime(); r.answer=raw.count(q.x0,q.x1,q.y0,q.y1); r.exact=System.nanoTime()-t;
        } else {
            long t=System.nanoTime(); Hist h=cache.equals("resident")?resident.get(q.histKey()):load(q); r.load=System.nanoTime()-t;
            t=System.nanoTime(); double estimate=h.histogram.RangeQuery(q.x0,q.x1,q.y0,q.y1); r.estimate=System.nanoTime()-t;
            Model m=models.get(q.seed+":"+method);
            t=System.nanoTime(); double[] x=values(m,h,q); r.features=System.nanoTime()-t;
            t=System.nanoTime(); r.log=m.booster.predict(x); r.error=Math.exp(r.log); r.inference=System.nanoTime()-t;
            t=System.nanoTime(); r.cutLog=cut(q,method); r.cutError=transformedCut(r.cutLog);
            r.accepted=Double.isFinite(r.log)&&r.error<=r.cutError; r.decision=System.nanoTime()-t;
            if(r.accepted) r.answer=estimate;
            else {t=System.nanoTime(); r.answer=raw.count(q.x0,q.x1,q.y0,q.y1); r.exact=System.nanoTime()-t;}
        }
        r.total=System.nanoTime()-start; return r;
    }
    static void verifyParity(List<Query> cases,Path out,double tol) throws Exception {
        boolean failed=false;
        try(PrintWriter w=new PrintWriter(Files.newBufferedWriter(out.resolve("parity.csv"),StandardCharsets.UTF_8))) {
            w.println("seed,qId,mergeRate,variant,python_score_log,java_score_log,cutoff_log,cutoff_error,score_close,python_accepted,java_log_accepted,java_error_accepted,decision_match,python_estimate,java_estimate,estimate_match");
            for(Query q:cases) for(String method:new String[]{A,B}) {
                Model m=models.get(q.seed+":"+method); double expected=q.expected(method),score=m.booster.predict(values(m,resident.get(q.histKey()),q)),c=cut(q,method);
                double estimate=resident.get(q.histKey()).histogram.RangeQuery(q.x0,q.x1,q.y0,q.y1);
                boolean estimateMatch=Double.isFinite(q.expectedEstimate)&&Double.isFinite(estimate)&&Math.abs(estimate-q.expectedEstimate)<=1e-6+1e-8*Math.abs(q.expectedEstimate);
                boolean close=Double.isFinite(expected)&&Double.isFinite(score)&&Math.abs(score-expected)<=tol*Math.max(1,Math.abs(expected));
                boolean py=expected<=c,log=score<=c,err=Math.exp(score)<=transformedCut(c),match=py==log&&py==err;
                w.println(q.seed+","+q.id+","+q.rate+","+method+","+expected+","+score+","+c+","+transformedCut(c)+","+close+","+py+","+log+","+err+","+match+","+q.expectedEstimate+","+estimate+","+estimateMatch);
                failed|=!close||!match||!estimateMatch;
            }
        }
        if(failed) throw new IllegalStateException("Python/Java score, estimate or acceptance mismatch; inspect parity.csv. Do not alter test gates to force a pass.");
    }
    static void verifyRaw(List<Query> cases,List<Query> warmup) {
        long t=System.nanoTime(); Map<String,Double> checked=new HashMap<>();
        List<Query> all=new ArrayList<>(cases); all.addAll(warmup);
        for(Query q:all) {
            String key=q.x0+":"+q.x1+":"+q.y0+":"+q.y1;
            Double prior=checked.get(key);
            if(prior!=null&&prior!=q.truth) throw new IllegalStateException("Inconsistent truth for one rectangle "+q.id);
            if(prior==null) {
                if(raw.count(q.x0,q.x1,q.y0,q.y1)!=q.truth) throw new IllegalStateException("Raw count differs from saved query truth for qId="+q.id+". Wrong raw data or query semantics.");
                checked.put(key,q.truth);
            }
        }
        setupRow("raw_truth_validation",-1,"all",Double.NaN,"resident",System.nanoTime()-t,"outside timed requests; "+checked.size()+" unique rectangles");
    }
    public static void main(String[] args) throws Exception {
        if(args.length!=1) throw new IllegalArgumentException("Usage: SourceInformationBenchmark CONFIG.properties");
        try(Reader r=Files.newBufferedReader(Paths.get(args[0]),StandardCharsets.UTF_8)){config.load(r);}
        Path output=Paths.get(required("output")); Files.createDirectories(output);
        for(String name:new String[]{"measurements.csv","parity.csv","setup.csv"}) if(Files.exists(output.resolve(name))) throw new IllegalArgumentException("Refusing to overwrite "+output.resolve(name));
        try(PrintWriter writer=new PrintWriter(Files.newBufferedWriter(output.resolve("setup.csv"),StandardCharsets.UTF_8))) {
            setup=writer; setup.println("stage,seed,variant,mergeRate,cache_mode,elapsed_ns,detail");
            List<Query> cases=queries(required("cases")),warmup=queries(required("warmup")); if(cases.isEmpty()) throw new IllegalArgumentException("No benchmark cases");
            sources=paths(required("sources"));
            long t=System.nanoTime(); raw=loadRaw(paths(required("raw_files")));
            if(raw.size()==0) throw new IllegalArgumentException("Empty raw data");
            setupRow("raw_load",-1,"all",Double.NaN,"resident",System.nanoTime()-t,raw.size()+" raw points");
            for(Map<String,String> r:csv(required("models"))) {
                int seed=Integer.parseInt(r.get("seed")); String v=r.get("variant");
                if(!v.equals(A)&&!v.equals(B)) throw new IllegalArgumentException("Unknown model variant "+v);
                t=System.nanoTime(); Model m=new Model(r.get("modelPath"),v);
                if(models.put(seed+":"+v,m)!=null) throw new IllegalArgumentException("Duplicate model");
                setupRow("model_load",seed,v,Double.NaN,"resident",System.nanoTime()-t,m.features.length+" features");
            }
            for(Map<String,String> r:csv(required("gates"))) {
                String k=Integer.parseInt(r.get("seed"))+":"+Double.toString(number(r.get("mergeRate")))+":"+r.get("variant");
                if(gates.put(k,number(r.get("cutoff_log")))!=null) throw new IllegalArgumentException("Duplicate gate "+k);
            }
            List<Query> all=new ArrayList<>(cases); all.addAll(warmup);
            for(Query q:all) {
                if(!models.containsKey(q.seed+":"+A)||!models.containsKey(q.seed+":"+B)) throw new IllegalArgumentException("Missing model for seed "+q.seed);
                cut(q,A);cut(q,B);
                if(!resident.containsKey(q.histKey())) {t=System.nanoTime(); resident.put(q.histKey(),load(q)); setupRow("histogram_context_load",-1,"all",q.rate,"resident",System.nanoTime()-t,q.arm);}
            }
            t=System.nanoTime(); verifyParity(cases,output,Double.parseDouble(config.getProperty("score_tolerance","1e-6")));
            setupRow("score_gate_validation",-1,"all",Double.NaN,"resident",System.nanoTime()-t,"outside timed requests"); verifyRaw(cases,warmup);
            int repeats=Integer.parseInt(config.getProperty("repeats","3")); if(repeats<1) throw new IllegalArgumentException("repeats must be positive");
            Random rng=new Random(Long.parseLong(config.getProperty("random_seed","0")));
            List<String> caches=new ArrayList<>(Arrays.asList(config.getProperty("cache_modes","resident").split(","))); Collections.shuffle(caches,rng);
            if(new HashSet<>(caches).size()!=caches.size()) throw new IllegalArgumentException("Duplicate cache modes");
            try(PrintWriter w=new PrintWriter(Files.newBufferedWriter(output.resolve("measurements.csv"),StandardCharsets.UTF_8))) {
                w.println("seed,qId,mergeRate,cache_mode,repeat,method,accepted,answer,truth,score_log,cutoff_log,score_error,cutoff_error,histogram_load_ns,estimate_ns,features_ns,inference_ns,decision_ns,exact_ns,total_ns");
                for(String cache:caches) {
                    if(!cache.equals("resident")&&!cache.equals("reload")) throw new IllegalArgumentException("cache_modes must be resident or reload");
                    t=System.nanoTime(); for(Query q:warmup) for(String m:METHODS) execute(q,m,cache);
                    setupRow("warmup",-1,"all",Double.NaN,cache,System.nanoTime()-t,warmup.size()+" calibration cases per method");
                    for(int rep=0;rep<repeats;rep++) {
                        List<Query> order=new ArrayList<>(cases);Collections.shuffle(order,rng);
                        for(Query q:order) {
                            List<String> methods=new ArrayList<>(Arrays.asList(METHODS));Collections.shuffle(methods,rng);
                            for(String method:methods) {
                                Result r=execute(q,method,cache);
                                // Validation and output are deliberately outside the request timer.
                                if(!r.accepted&&r.answer!=q.truth) throw new IllegalStateException("Exact result mismatch "+q.id);
                                if(!method.equals("always_exact")&&r.accepted!=(q.expected(method)<=r.cutLog)) throw new IllegalStateException("Gate decision changed after validation "+q.id);
                                w.println(q.seed+","+q.id+","+q.rate+","+cache+","+rep+","+method+","+r.accepted+","+r.answer+","+q.truth+","+r.log+","+r.cutLog+","+r.error+","+r.cutError+","+r.load+","+r.estimate+","+r.features+","+r.inference+","+r.decision+","+r.exact+","+r.total);
                            }
                        }
                        w.flush();System.out.println("Completed "+cache+" repeat "+(rep+1)+"/"+repeats+" ("+cases.size()+" cases x 3 methods)");
                    }
                }
            }
        }
    }
}

"""

## Execute

Run this cell once after updating widgets. Each execution creates a new result directory and preserves failures for
inspection. A missing 08d run path, missing weights, incompatible JAR, score mismatch or raw-count mismatch stops
execution. Results are never invented or silently replaced with another model/backend.

In [ ]:
# RUN_CELL
config = load_configuration()
result_dir, summary = run_benchmark(config)
display(summary) if 'display' in globals() else print(summary.to_string(index=False))

## Read the outputs

- `gate_audit.csv`: full saved-test cohort counts, reconstructed without changing thresholds.
- `per_query.csv`: every timed request, original query ID, score/cutoff, route, answer, quality and all stage times.
- `quality_cases.csv`: one quality observation per query/setting/method, without timing repeats.
- `summary.csv`: mean/p95 latency, accepted/violating counts and rate, speedup versus the matching exact baseline.
- `paired_timings.csv`: matched exact/A/B request observations. The p95 speedup field is a ratio of marginal p95s.
- `setup.csv`, `run.json`: initialization, identities, original training metadata, protocol and execution status.
- Each policy folder also has `parity.csv` and an execution log. `e2e_results.zip` exports compact results without models/raw data.

Zero acceptance has undefined risk, not zero risk. Check answer quality alongside speed: an above-target method is
not a quality-matched speed improvement. Seeds/rates repeat query IDs; no pooled binomial claims or significance
tests are made. Full-cohort offline quality and sampled timed quality are distinct outputs. Existing 08d test data
has been inspected previously: this remains exploratory and does not replace fresh holdout validation.

In [ ]:
# PLOT_CELL
import matplotlib.pyplot as plt

plot = summary.copy()
plot['setting'] = plot.apply(lambda r: f"seed{r.seed} / rate{r.mergeRate:.3f} / {r.policy} / {r.cache_mode}", axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for method, group in plot.groupby('method', sort=False):
    axes[0].scatter(group.acceptance * 100, group.total_mean_ms, label=method, alpha=0.75)
    if method != 'always_exact':
        axes[1].scatter(group.acceptance * 100, group.accepted_violation_rate * 100, label=method, alpha=0.75)
axes[0].set(xlabel='Accepted from histogram (%)', ylabel='Mean request latency (ms)', title='Latency and acceptance by setting')
axes[1].axhline(5, color='black', linestyle='--', label='5% target')
axes[1].set(xlabel='Accepted from histogram (%)', ylabel='Violation among accepted (%)', title='Quality must accompany speed')
for axis in axes:
    axis.legend(fontsize=7)
    axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(Path(result_dir) / 'latency_quality.png', dpi=160)
export_results(Path(result_dir))
plt.show()
print('Results:', result_dir)
print('Upload e2e_results.zip for analysis; model weights and raw points are not included.')

## Diagnose a failed parity check
Run only the cell below after a parity error. It exports the selected queries, saved weights, histogram summaries and actual runtime JAR for reproducible diagnosis. It does not retrain, change cutoffs, rerun the benchmark or include raw point files. Leave the path blank to select the most recent failed parity run under `output_root`.


In [ ]:
# Run ONLY this cell after a parity failure. No training, calibration or benchmark is run.
# Leave blank to select the most recent failed parity run under the output_root widget.
DIAGNOSTIC_RUN_DIR = ""

import csv as _debug_csv
import hashlib as _debug_hashlib
import json as _debug_json
from pathlib import Path as _DebugPath
from zipfile import ZipFile as _DebugZipFile, ZIP_DEFLATED as _DEBUG_DEFLATED


def _export_parity_debug(run_path=""):
    if run_path.strip():
        failed = _DebugPath(run_path).expanduser().resolve()
    else:
        try:
            root = _DebugPath(dbutils.widgets.get('output_root'))
        except NameError:
            root = _DebugPath(get_setting('output_root'))
        candidates = []
        for path in root.glob('e2e-source-information-*/run.json'):
            metadata = _debug_json.loads(path.read_text())
            if (metadata.get('status') == 'failed'
                    and metadata.get('study') == 'e2e-source-information'
                    and list(path.parent.glob('*/parity.csv'))):
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('No failed parity run found. Set DIAGNOSTIC_RUN_DIR explicitly.')
        failed = max(candidates, key=lambda p: p.stat().st_mtime_ns).parent
    print('Diagnostic source:', failed)
    state = _debug_json.loads((failed / 'run.json').read_text())
    stage = _DebugPath(state['staging_directory'])
    if not (stage / 'cases.csv').is_file():
        raise FileNotFoundError('Temporary inputs have expired. Rerun the benchmark, then run only this diagnostic cell.')
    items = {}

    def add(path, name):
        path = _DebugPath(path)
        if not path.is_file():
            raise FileNotFoundError('Missing diagnostic input: ' + str(path))
        items[name] = path

    add(failed / 'run.json', 'run.json')
    for name in ('cases.csv', 'models.csv', 'sources.txt', 'SourceInformationBenchmark.java'):
        add(stage / name, 'adapter/' + name)
    for path in stage.glob('gates-*.csv'):
        add(path, 'adapter/' + path.name)
    for path in failed.glob('*/parity.csv'):
        add(path, path.parent.name + '/parity.csv')
    for path in failed.glob('*/execution.log'):
        add(path, path.parent.name + '/execution.log')
    for path in (stage / 'sources.txt').read_text().splitlines():
        if path.strip():
            add(path, 'histograms/' + _DebugPath(path).name)
    with (stage / 'cases.csv').open() as stream:
        for row in _debug_csv.DictReader(stream):
            add(row['armPath'], 'histograms/' + _DebugPath(row['armPath']).name)
    with (stage / 'models.csv').open() as stream:
        for row in _debug_csv.DictReader(stream):
            add(row['modelPath'], 'models/' + _DebugPath(row['modelPath']).name)
    jar = _DebugPath(state['config']['jar_path'])
    if _debug_hashlib.sha256(jar.read_bytes()).hexdigest() != state['jar_sha256']:
        raise ValueError('Runtime JAR changed since failure; preserve the original JAR before diagnosis.')
    add(jar, 'runtime/synopsislake.jar')
    repo = _DebugPath(state['config']['repo_root'])
    for name in ('features.py', 'features_v1.py'):
        add(repo / 'ml' / name, 'feature_code/' + name)
    destination = failed / 'parity_debug.zip'
    with _DebugZipFile(destination, 'w', _DEBUG_DEFLATED) as archive:
        for name, path in sorted(items.items()):
            archive.write(path, name)
        archive.writestr('README.txt',
            'Diagnostic only: selected queries, saved model weights, histogram summaries, runtime JAR and feature code.\n'
            'Original raw point files and authentication credentials are NOT included.\n'
            'Expected Python scores in adapter/cases.csv were checked against saved 08d predictions.\n'
            'Current feature-code identity can be compared with run.json feature_code_sha256.\n')
    print('Created:', destination)
    print('Contains model weights and histogram summaries; no raw point files. Upload this ZIP for diagnosis.')
    return destination


_debug_latest = globals().get('_benchmark_last_result_dir')
if (not DIAGNOSTIC_RUN_DIR.strip() and _debug_latest
        and _debug_json.loads((_DebugPath(_debug_latest) / 'run.json').read_text()).get('status') == 'complete'):
    print('Latest benchmark completed successfully; no failure diagnostic export needed.')
else:
    parity_debug_zip = _export_parity_debug(DIAGNOSTIC_RUN_DIR)
